## 1. Libraries

In [ ]:
import os
import time
import random
import warnings
import logging
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch

from pathlib import Path
from IPython.display import display
from neuralforecast import NeuralForecast
from neuralforecast.models import KAN, NHITS, NBEATSx, GRU, LSTM, MLP
from neuralforecast.losses.pytorch import MAE

warnings.filterwarnings("ignore")
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

## 2. Load datasets

In [ ]:
base_path = Path.cwd()

datasets = {
    "HUPA": "HUPA_UCM_IOB_COB.parquet",
    "AZT1D": "AZT1D_IOB_COB.parquet"
}

data_all = {}

for dataset, filename in datasets.items():
    df = pd.read_parquet(base_path / filename)
    df = df.rename(columns={"time": "ds", "glucose": "y"})
    df["ds"] = pd.to_datetime(df["ds"])
    data = (df.copy().sort_values(["unique_id", "ds"])
            .reset_index(drop=True))
    data_all[dataset] = data

    print("\nDataset:", dataset)
    print("Rows:", len(data))
    print("Patients:", data["unique_id"].nunique())
    display(data.head())

## 3. Historical exogenous variables and experiments

In [ ]:
insulin_cols = ["basal_rate", "bolus_volume_delivered"]
iob_cols = [f"IOB_DIA_{DIA}h" for DIA in range(2, 9)]
activity_cols = ["heart_rate", "steps", "calories"]

stat_exog_list = []

experiments = {
    "E01": [],
    "E02": insulin_cols,
    "E03": ["carb_input"],
    "E04": ["COB"],
    "E05": ["IOB_DIA_2h"],
    "E06": ["IOB_DIA_3h"],
    "E07": ["IOB_DIA_4h"],
    "E08": ["IOB_DIA_5h"],
    "E09": ["IOB_DIA_6h"],
    "E10": ["IOB_DIA_7h"],
    "E11": ["IOB_DIA_8h"],
    "E12": iob_cols,
    "E13": insulin_cols + ["carb_input"],
    "E14": insulin_cols + ["COB"],
    "E15": iob_cols + ["carb_input"],
    "E16": iob_cols + ["COB"],
    "E17": insulin_cols + ["COB"] + activity_cols,
    "E18": iob_cols + ["COB"] + activity_cols
}

for name, features in experiments.items():
    print(f"{name}: CGM + {features}")

## 4. Experimental configuration

In [ ]:
# Model selection: KAN, NHITS, NBEATSx, GRU, LSTM, MLP
selected_models = ["GRU"]

# Cross-validation configuration
freq = "5min"
horizons = [6, 12, 18, 24]
step_sizes = [6, 12, 18, 24]
evaluation_hours = 24
evaluation_size = evaluation_hours * 60 // 5
val_size = 0

cv_config = {}

for horizon, step_size in zip(horizons, step_sizes):
    n_windows = (evaluation_size - horizon) // step_size + 1
    cv_config[horizon] = {
        "step_size": step_size,
        "n_windows": n_windows
    }

display(pd.DataFrame(cv_config).T.rename_axis("horizon"))

# Training configuration
random_seed = 42
max_steps = 200
USE_AZT1D_BASAL = False

np.random.seed(random_seed)
random.seed(random_seed)
torch.manual_seed(random_seed)
torch.cuda.manual_seed_all(random_seed)

output_path = base_path / "results_ablation" / "LOCAL"
output_path.mkdir(parents=True, exist_ok=True)

## 5. Prepare local datasets

In [ ]:
local_data = {}
available_experiments = {}

for dataset, data in data_all.items():
    available_experiments[dataset] = {
        experiment: features
        for experiment, features in experiments.items()
        if all(c in data.columns for c in features)
    }

    # AZT1D basal is excluded until its temporal alignment is verified.
    if dataset == "AZT1D" and not USE_AZT1D_BASAL:
        available_experiments[dataset] = {
            experiment: features
            for experiment, features in available_experiments[dataset].items()
            if "basal_rate" not in features
        }

    common_features = sorted({
        c for features in available_experiments[dataset].values()
        for c in features
    })

    local_data[dataset] = {}

    for patient in sorted(data["unique_id"].unique()):
        df_patient = (
            data.loc[data["unique_id"] == patient]
            .copy().sort_values("ds").reset_index(drop=True)
        )

        breaks = df_patient["ds"].diff().ne(pd.Timedelta(freq))
        valid = pd.Series(
            np.isfinite(df_patient[["y"] + common_features]
                        .to_numpy(dtype=float)).all(axis=1),
            index=df_patient.index
        )

        # Conservative margin for incomplete event history in AZT1D.
        if dataset == "AZT1D":
            event_missing = df_patient[
                ["bolus_volume_delivered", "carb_input"]
            ].isna().any(axis=1)
            last_reset = df_patient["ds"].where(breaks | event_missing).ffill()
            elapsed = df_patient["ds"] - last_reset
            valid &= elapsed.ge(pd.Timedelta(hours=8))

        df_patient["block_id"] = (breaks | ~valid).cumsum()

        blocks = [
            block for _, block in df_patient.loc[valid].groupby("block_id")
            if len(block) >= evaluation_size + 36 + max(horizons)
        ]

        # Most recent eligible block, shared by all experiments and horizons.
        if blocks:
            local_data[dataset][patient] = (
                blocks[-1].drop(columns="block_id").reset_index(drop=True)
            )

    print("\nDataset:", dataset)
    print("Patients:", len(local_data[dataset]))
    print("Experiments:", len(available_experiments[dataset]))

## 6. Model configuration

In [ ]:
def create_models(horizon, input_size, hist_exog_list, model_name):
    common_kwargs = dict(
        h=horizon,
        input_size=input_size,
        futr_exog_list=[],
        hist_exog_list=hist_exog_list,
        stat_exog_list=stat_exog_list,
        loss=MAE(),
        max_steps=max_steps,
        learning_rate=1e-3,
        scaler_type="standard",
        random_seed=random_seed,
        batch_size=16,
        valid_batch_size=16,
        start_padding_enabled=True,
        val_check_steps=50,
        early_stop_patience_steps=-1,
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
        accelerator="gpu",
        devices=1
    )

    models = []

    if model_name == "KAN":
        models.append(KAN(**common_kwargs, hidden_size=512,
                          n_hidden_layers=1, alias="KAN"))
    elif model_name == "NHITS":
        models.append(NHITS(**common_kwargs, n_blocks=[1, 1, 1],
                            mlp_units=[[512, 512], [512, 512], [512, 512]],
                            alias="NHITS"))
    elif model_name == "NBEATSx":
        models.append(NBEATSx(**common_kwargs,
                              stack_types=["identity", "trend", "seasonality"],
                              n_blocks=[1, 1, 1],
                              mlp_units=[[512, 512], [512, 512], [512, 512]],
                              alias="NBEATSx"))
    elif model_name == "GRU":
        models.append(GRU(**common_kwargs, encoder_n_layers=2,
                          encoder_hidden_size=200, decoder_hidden_size=128,
                          decoder_layers=2, alias="GRU"))
    elif model_name == "LSTM":
        models.append(LSTM(**common_kwargs, encoder_n_layers=2,
                           encoder_hidden_size=200, decoder_hidden_size=128,
                           decoder_layers=2, alias="LSTM"))
    elif model_name == "MLP":
        models.append(MLP(**common_kwargs, hidden_size=512,
                          num_layers=2, alias="MLP"))

    return models

## 7. Format NeuralForecast forecasts

In [ ]:
def format_neuralforecast_forecasts(df_, sample_type, model_cols):
    df_ = df_.copy()
    df_["ds"] = pd.to_datetime(df_["ds"])
    df_["cutoff"] = pd.to_datetime(df_["cutoff"])

    id_cols = ["unique_id", "ds", "cutoff", "y"]
    out = df_.melt(id_vars=id_cols, value_vars=model_cols,
                   var_name="model", value_name="forecast")
    out["forecast"] = out["forecast"].clip(lower=0)
    out["type"] = sample_type
    out["frequency"] = freq

    if sample_type == "out_sample":
        out["horizon"] = ((out["ds"] - out["cutoff"]) / pd.Timedelta(freq)).astype(int)
    else:
        out["horizon"] = np.nan

    return out

## 8. Automatic local training

In [ ]:
for model_name in selected_models:
    for dataset in local_data:
        patients = list(local_data[dataset].keys())

        for patient in patients:
            for experiment, hist_exog_list in available_experiments[dataset].items():
                for horizon in horizons:

                    # Cross-validation configuration for this horizon
                    step_size = cv_config[horizon]["step_size"]
                    n_windows = cv_config[horizon]["n_windows"]

                    # Output directory and resume
                    result_path = (output_path / model_name / dataset / patient
                                   / experiment / f"H{horizon * 5}")
                    result_path.mkdir(parents=True, exist_ok=True)
                    prediction_file = result_path / "predictions.parquet"

                    if prediction_file.exists():
                        continue

                    # Local patient data
                    data_nf = (
                        local_data[dataset][patient][["unique_id", "ds", "y"] + hist_exog_list]
                        .copy().sort_values(["unique_id", "ds"]).reset_index(drop=True)
                    )

                    # Input size
                    series_lengths = data_nf.groupby("unique_id").size()
                    min_train_size = (
                        series_lengths.min() - horizon - (n_windows - 1) * step_size
                    )
                    input_size = min(36, max(12, min_train_size - 1))

                    print("\n" + "=" * 60)
                    print("Model:", model_name)
                    print("Dataset:", dataset)
                    print("Patient:", patient)
                    print("Experiment:", experiment)
                    print("Features:", hist_exog_list)
                    print("Horizon:", horizon * 5, "minutes")
                    print("Step size:", step_size)
                    print("Number of windows:", n_windows)
                    print("Input size:", input_size)
                    print("Training samples:", min_train_size)

                    # Create a fresh, independent local model
                    models = create_models(horizon, input_size, hist_exog_list, model_name)
                    nf = NeuralForecast(models=models, freq=freq)

                    # NeuralForecast cross-validation
                    t0 = time.perf_counter()
                    cv_raw = nf.cross_validation(
                        df=data_nf,
                        n_windows=n_windows,
                        step_size=step_size,
                        val_size=val_size,
                        refit=True,
                        verbose=True,
                        id_col="unique_id",
                        time_col="ds",
                        target_col="y"
                    )
                    cv_time = time.perf_counter() - t0
                    cv_raw["ds"] = pd.to_datetime(cv_raw["ds"])
                    cv_raw["cutoff"] = pd.to_datetime(cv_raw["cutoff"])

                    # Format predictions
                    model_names = [m.alias for m in models]
                    cv_out = format_neuralforecast_forecasts(
                        df_=cv_raw, sample_type="out_sample", model_cols=model_names
                    )
                    cv_cutoffs = (cv_out["cutoff"].drop_duplicates()
                                  .sort_values().reset_index(drop=True))

                    pred_all = cv_out.assign(
                        dataset=dataset,
                        experiment=experiment,
                        training_type="local",
                        cv_time_seconds=cv_time,
                        horizon_config=horizon,
                        n_windows=n_windows,
                        step_size=step_size,
                        input_size=input_size,
                        max_steps=max_steps,
                        features=",".join(["CGM"] + hist_exog_list),
                        static_features=",".join(stat_exog_list),
                        train_start=data_nf["ds"].min(),
                        initial_cutoff=cv_cutoffs.min(),
                        test_end=cv_raw["ds"].max()
                    )
                    pred_all["prediction_minutes"] = pred_all["horizon"] * 5
                    pred_all["horizon_minutes"] = horizon * 5

                    ordered_cols = [
                        "dataset", "experiment", "unique_id", "ds", "cutoff", "y",
                        "forecast", "model", "type", "frequency", "horizon",
                        "prediction_minutes", "horizon_config", "horizon_minutes",
                        "training_type", "cv_time_seconds", "n_windows", "step_size",
                        "input_size", "max_steps", "features", "static_features",
                        "train_start", "initial_cutoff", "test_end"
                    ]
                    pred_all = pred_all[ordered_cols]

                    # Save predictions
                    temp_file = result_path / "predictions.tmp.parquet"
                    pred_all.to_parquet(temp_file, index=False)
                    os.replace(temp_file, prediction_file)
                    print(f"CV completed in {cv_time:.2f} seconds")
                    print("Predictions saved.")

                    # Release memory before the next training
                    del nf, models, cv_raw, cv_out, pred_all
                    gc.collect()
                    torch.cuda.empty_cache()

## 9. Consolidate predictions

In [ ]:
prediction_files = sorted(output_path.rglob("predictions.parquet"))
pred_all = pd.concat([pd.read_parquet(file) for file in prediction_files],
                     ignore_index=True)
display(pred_all.head())
print("Prediction files:", len(prediction_files))
print("Total rows:", len(pred_all))
print(pred_all["model"].value_counts())
pred_all.to_parquet(output_path / "predictions_all.parquet", index=False)

## 10. Overall evaluation metrics

In [ ]:
out_df = pred_all.query("type == 'out_sample'").copy()
out_df["absolute_error"] = np.abs(out_df["y"] - out_df["forecast"])
out_df["squared_error"] = (out_df["y"] - out_df["forecast"]) ** 2

metrics = (
    out_df.groupby(["dataset", "unique_id", "experiment", "model", "horizon_minutes"])
    .agg(MAE=("absolute_error", "mean"), MSE=("squared_error", "mean"),
         samples=("y", "size"))
    .reset_index()
)
metrics["RMSE"] = np.sqrt(metrics.pop("MSE"))
display(metrics.round(3))
metrics.to_csv(output_path / "metrics.csv", index=False)

## 11. Metrics by prediction horizon

In [ ]:
metrics_horizon = (
    out_df.groupby(["dataset", "unique_id", "experiment", "model",
                    "horizon_minutes", "prediction_minutes"])
    .agg(MAE=("absolute_error", "mean"), MSE=("squared_error", "mean"),
         samples=("y", "size"))
    .reset_index()
)
metrics_horizon["RMSE"] = np.sqrt(metrics_horizon.pop("MSE"))
display(metrics_horizon.round(3))
metrics_horizon.to_csv(output_path / "metrics_horizon.csv", index=False)

metrics_final = metrics_horizon.loc[
    metrics_horizon["prediction_minutes"] == metrics_horizon["horizon_minutes"]
].copy()
display(metrics_final.round(3))
metrics_final.to_csv(output_path / "metrics_final.csv", index=False)

## 12. Plot forecasts

In [ ]:
# Visualization only: select an existing patient / experiment / horizon.
dataset = "HUPA"
patient = "HUPA0001P"
experiment = "E16"
horizon = 6

s = out_df.loc[
    (out_df["dataset"] == dataset) &
    (out_df["unique_id"] == patient) &
    (out_df["experiment"] == experiment) &
    (out_df["horizon_config"] == horizon)
].copy()

model_colors = {"NHITS": "blue", "NBEATSx": "red", "KAN": "green",
                "GRU": "purple", "LSTM": "orange", "MLP": "brown"}
fig, ax = plt.subplots(figsize=(16, 5))
actual = s.groupby("ds", as_index=False)["y"].first().sort_values("ds")
ax.plot(actual["ds"], actual["y"], color="black", linewidth=2, label="Actual")
for model, g in s.groupby("model"):
    g = g.sort_values("ds")
    ax.plot(g["ds"], g["forecast"], color=model_colors.get(model, "gray"),
            linestyle="--", linewidth=1.8, label=model)
ax.axhline(70, color="red", linestyle=":", alpha=0.7)
ax.axhline(180, color="orange", linestyle=":", alpha=0.7)
ax.set_title(f"{patient} — {experiment} — {horizon * 5} min", fontweight="bold")
ax.set_ylabel("Glucose (mg/dL)")
ax.set_xlabel("Time")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
ax.grid(True, alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend()
plt.tight_layout()
plt.show()

## 13. Plot MAE and RMSE

In [ ]:
s_metrics = metrics_horizon.loc[
    (metrics_horizon["dataset"] == dataset) &
    (metrics_horizon["unique_id"] == patient) &
    (metrics_horizon["experiment"] == experiment) &
    (metrics_horizon["horizon_minutes"] == horizon * 5)
].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for model, g in s_metrics.groupby("model"):
    g = g.sort_values("prediction_minutes")
    axes[0].plot(g["prediction_minutes"], g["MAE"],
                 color=model_colors.get(model, "gray"), marker="o", label=model)
    axes[1].plot(g["prediction_minutes"], g["RMSE"],
                 color=model_colors.get(model, "gray"), marker="o", label=model)
axes[0].set_title("MAE")
axes[0].set_ylabel("MAE (mg/dL)")
axes[1].set_title("RMSE")
axes[1].set_ylabel("RMSE (mg/dL)")
for ax in axes:
    ax.set_xlabel("Prediction horizon (min)")
    ax.grid(True, alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend()
fig.suptitle(f"{patient} — {experiment} — {horizon * 5} min", fontweight="bold")
plt.tight_layout()
plt.show()